# v2_07_scibert_plus_donnees : SciBERT avec plus d'articles d'entraînement

Même code que `v2_06_scibert_optim` (mêmes hyperparamètres que v2_05 : 4 epochs, lr 2e-5, max_length 384, lot 16 x 2 GPU).
**Seul changement :** le train reçoit en plus `train_plus.csv` (~20 000 articles tirés par `v2_02_train_plus`).
Validation et test sont **identiques** -> comparaison juste avec v2_05 (val micro-F1 0,8966).

In [ ]:
# ===== RÉGLAGES DE L'ESSAI : on ne change qu'UNE chose à la fois =====
ESSAI = "plus_donnees"   # nom de l'essai (utilisé dans les noms de fichiers)
TRAIN_PLUS = True        # True = train + train_plus.csv (~20 000 articles en plus)
NB_EPOCHS = 4            # essai epochs6 : 6 n'apporte rien (surapprentissage après la 4e) -> on garde 4
LEARNING_RATE = 2e-5     # v2_05 : 2e-5
MAX_LENGTH = 384         # v2_05 : 384
TAILLE_LOT = 16          # v2_05 : 16 (par GPU)

In [ ]:
import glob, os, sys, math, json
import numpy as np
import pandas as pd
import torch
import transformers

DOSSIER = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
sys.path.append(os.path.dirname(glob.glob("/kaggle/input/**/metrics.py", recursive=True)[0]))
from metrics import GROUPES, evaluer, rapport_par_groupe, probas_vers_predictions, chercher_seuils

train = pd.read_csv(f"{DOSSIER}/train.csv", dtype={"id": str})
val = pd.read_csv(f"{DOSSIER}/val.csv", dtype={"id": str})
test = pd.read_csv(f"{DOSSIER}/test.csv", dtype={"id": str})

if TRAIN_PLUS:   # on ajoute les articles en plus au TRAIN seulement (val et test inchangés)
    train_plus = pd.read_csv(glob.glob("/kaggle/input/**/train_plus.csv", recursive=True)[0], dtype={"id": str})
    communs = set(train_plus["id"]) & (set(train["id"]) | set(val["id"]) | set(test["id"]))
    assert len(communs) == 0, f"{len(communs)} articles de train_plus déjà présents ailleurs !"
    print("Articles ajoutés au train :", len(train_plus))
    train = pd.concat([train, train_plus], ignore_index=True)

print("Essai :", ESSAI)
print("Train :", train.shape, "| Validation :", val.shape, "| Test :", test.shape)
print("GPU :", torch.cuda.device_count(), "x",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN")

In [ ]:
from datasets import Dataset, Sequence, Value
from transformers import AutoTokenizer

NOM_MODELE = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(NOM_MODELE)

def vers_dataset(df, avec_labels=True):
    donnees = {"text": df["text"].tolist()}
    if avec_labels:
        donnees["labels"] = df[GROUPES].values.astype("float32").tolist()
    ds = Dataset.from_dict(donnees)
    if avec_labels:
        ds = ds.cast_column("labels", Sequence(Value("float32")))
    return ds.map(lambda lot: tokenizer(lot["text"], truncation=True, max_length=MAX_LENGTH),
                  batched=True, remove_columns=["text"])

ds_train = vers_dataset(train)
ds_val = vers_dataset(val)
ds_test = vers_dataset(test, avec_labels=False)   # sans labels -> aucun score test possible
print(ds_train)

In [ ]:
from transformers import AutoModelForSequenceClassification

modele = AutoModelForSequenceClassification.from_pretrained(
    NOM_MODELE, num_labels=len(GROUPES), problem_type="multi_label_classification",
    id2label=dict(enumerate(GROUPES)), label2id={g: i for i, g in enumerate(GROUPES)})

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

def calculer_metriques(eval_pred):
    logits, labels = eval_pred
    probas = 1 / (1 + np.exp(-logits))
    return evaluer(labels.astype(int), probas_vers_predictions(probas))

nb_gpus = max(1, torch.cuda.device_count())
pas_par_epoch = math.ceil(len(ds_train) / (TAILLE_LOT * nb_gpus))
pas_warmup = int(0.1 * pas_par_epoch * NB_EPOCHS)
print("Pas par epoch :", pas_par_epoch, "| pas de warmup :", pas_warmup)

args = TrainingArguments(
    output_dir="/kaggle/working/scibert", num_train_epochs=NB_EPOCHS,
    learning_rate=LEARNING_RATE, per_device_train_batch_size=TAILLE_LOT, per_device_eval_batch_size=64,
    warmup_steps=pas_warmup, weight_decay=0.01, fp16=True,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="micro_f1", save_total_limit=1,
    logging_strategy="epoch", report_to="none", seed=42)

trainer = Trainer(model=modele, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=calculer_metriques)
trainer.train()

In [ ]:
pred_val = trainer.predict(ds_val)
probas_val = 1 / (1 + np.exp(-pred_val.predictions))
y_val = pred_val.label_ids.astype(int)

scores_05 = evaluer(y_val, probas_vers_predictions(probas_val))
seuils = chercher_seuils(y_val, probas_val)
scores_seuils = evaluer(y_val, probas_vers_predictions(probas_val, seuils))

meilleure_epoch = round(int(trainer.state.best_model_checkpoint.split("-")[-1]) / pas_par_epoch)
print("Essai :", ESSAI, "| meilleure epoch :", meilleure_epoch)
print("Seuils choisis :", dict(zip(GROUPES, seuils.tolist())))
print(f"{'':16s} {'seuil 0.5':>10s} {'seuils/groupe':>14s}")
for nom in scores_05:
    print(f"{nom:16s} {scores_05[nom]:>10.4f} {scores_seuils[nom]:>14.4f}")
print()
print(rapport_par_groupe(y_val, probas_vers_predictions(probas_val, seuils)))

reglages = {"NB_EPOCHS": NB_EPOCHS, "LEARNING_RATE": LEARNING_RATE,
            "MAX_LENGTH": MAX_LENGTH, "TAILLE_LOT": TAILLE_LOT,
            "TRAIN_PLUS": TRAIN_PLUS, "taille_train": len(train)}
with open(f"/kaggle/working/scibert_{ESSAI}_val.json", "w") as f:
    json.dump({"reglages": reglages, "meilleure_epoch": meilleure_epoch, "seuil_05": scores_05,
               "seuils": seuils.tolist(), "seuils_par_groupe": scores_seuils,
               "historique": trainer.state.log_history}, f, indent=2)
np.save(f"/kaggle/working/scibert_{ESSAI}_val_probas.npy", probas_val)
print(f"Sauvegardé : scibert_{ESSAI}_val.json, scibert_{ESSAI}_val_probas.npy")

In [ ]:
import shutil

pred_test = trainer.predict(ds_test)
probas_test = 1 / (1 + np.exp(-pred_test.predictions))
np.save(f"/kaggle/working/scibert_{ESSAI}_test_probas.npy", probas_test)
test[["id"]].to_csv(f"/kaggle/working/scibert_{ESSAI}_test_ids.csv", index=False)
print("Probabilités test sauvegardées :", probas_test.shape)

shutil.rmtree("/kaggle/working/scibert", ignore_errors=True)   # supprime les checkpoints